# Sales Forecasting - Interactive Visualization & Reporting

This notebook creates interactive dashboards and summary reports.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path('..').resolve()))

from src.visualization import (
    create_interactive_forecast,
    create_forecast_with_confidence,
    create_seasonal_decomposition_plot,
    create_monthly_sales_plot,
    create_forecast_dashboard,
    save_interactive_plot,
)
from src.models import (
    calculate_metrics,
    train_test_split_time_series,
    fit_sarima,
)
from src.time_series_analysis import identify_peak_periods

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Screenshot settings
SCREENSHOTS_DIR = Path('../reports/screenshots')
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

def save_screenshot(fig, name):
    filepath = SCREENSHOTS_DIR / f'{name}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f'Saved: {filepath}')

## 1. Load Data and Train Model

In [ ]:
df = pd.read_csv('../data/processed/daily_sales_features.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')
series = df['total_sales'].dropna()

train, test = train_test_split_time_series(series, test_size=0.2)

sarima_result = fit_sarima(
    train, 
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    forecast_steps=len(test)
)

metrics = calculate_metrics(test.values, sarima_result['predictions'].values)

print(f"Model: SARIMA(1,1,1)(1,1,1,7)")
print(f"MAPE: {metrics['mape']:.2f}%")
print(f"RMSE: £{metrics['rmse']:,.2f}")
print(f"MAE: £{metrics['mae']:,.2f}")

## 2. Interactive Forecast Plot

In [ ]:
fig_forecast = create_interactive_forecast(
    actual=test,
    predicted=sarima_result['predictions'],
    title=f'Sales Forecast - SARIMA Model (MAPE: {metrics["mape"]:.2f}%)'
)

output_path = save_interactive_plot(fig_forecast, 'interactive_forecast')
print(f"Saved interactive plot: {output_path}")

fig_forecast.show()

## 3. Monthly Sales Analysis

In [ ]:
df_reset = df.reset_index()
fig_monthly = create_monthly_sales_plot(df_reset)

output_path = save_interactive_plot(fig_monthly, 'monthly_sales')
print(f"Saved interactive plot: {output_path}")

fig_monthly.show()

## 4. Comprehensive Dashboard

In [ ]:
peak_periods = identify_peak_periods(df.reset_index(), date_col='Date', value_col='total_sales')

fig_dashboard = create_forecast_dashboard(
    actual=test,
    predicted=sarima_result['predictions'],
    metrics=metrics,
    seasonal_pattern=peak_periods['daily_pattern']
)

output_path = save_interactive_plot(fig_dashboard, 'forecast_dashboard')
print(f"Saved interactive plot: {output_path}")

fig_dashboard.show()

## 5. Executive Summary Report

In [ ]:
summary = """
============================================================
EXECUTIVE SUMMARY: Sales Forecasting Analysis
============================================================

OVERVIEW
--------
This report summarizes the sales forecasting analysis for the UCI Online Retail
dataset (541K transactions, Dec 2010 - Dec 2011).

KEY FINDINGS
------------
1. BUSINESS MODEL: B2B wholesale, UK-focused (80% of revenue)
2. SALES TREND: Growing - sales doubled over the year
3. SEASONALITY: Strong weekly pattern (Thu peak), December is busiest month
4. DATA QUALITY: Non-stationary, requires differencing for modeling

MODEL PERFORMANCE
-----------------
Model: SARIMA(1,1,1)(1,1,1,7)
MAPE: {mape:.2f}% (Target: <= 15%)
RMSE: £{rmse:,.2f}
MAE: £{mae:,.2f}

RECOMMENDATIONS
---------------
1. INVENTORY: Stock up 2-3 weeks before December peak
2. STAFFING: Increase capacity on Tuesday/Thursday
3. MARKETING: Run promotions on slower days (Monday/Wednesday)
4. EXPANSION: Explore growth in Ireland and Germany

NEXT STEPS
----------
1. Deploy model for weekly automated forecasting
2. Add external features (holidays, promotions)
3. Test ensemble methods (combine ARIMA with ML models)
4. Build real-time dashboard for stakeholders

============================================================
""".format(mape=metrics['mape'], rmse=metrics['rmse'], mae=metrics['mae'])

print(summary)

report_path = Path('../reports/executive_summary.txt')
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w') as f:
    f.write(summary)
print(f"Saved report: {report_path}")

## 6. Export Forecast Data

In [ ]:
forecast_df = pd.DataFrame({
    'date': test.index,
    'actual': test.values,
    'predicted': sarima_result['predictions'].values,
    'residual': test.values - sarima_result['predictions'].values
})

output_path = Path('../data/processed/forecast_results.csv')
forecast_df.to_csv(output_path, index=False)
print(f"Saved forecast results: {output_path}")

print(f"\nForecast Summary:")
print(f"- Total predictions: {len(forecast_df)}")
print(f"- Date range: {forecast_df['date'].min()} to {forecast_df['date'].max()}")
print(f"- Mean actual: £{forecast_df['actual'].mean():,.2f}")
print(f"- Mean predicted: £{forecast_df['predicted'].mean():,.2f}")